# The Goal is to estimate surface NO2 concentrations with Machine Learning because sensors are of high cost. 

In [1]:
import pandas as pd 
import numpy as np
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,mean_absolute_percentage_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

## Data Loading

### the data is

- Ground-truth data from air quality monitoring stations in the continental part of Lombardy and Veneto regions.
- Remote sensing data of NO2 from Sentinel-5P TROPOMI, precipitation from CHIRPS and land surface temperature from NOAA datasets, all from Google Earth Engine (GEE).

In [2]:
train = pd.read_csv('Train.csv')

In [3]:
train.info() # there are data points without a target

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86584 entries, 0 to 86583
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Zindi            86584 non-null  object 
 1   Date                86584 non-null  object 
 2   ID                  86584 non-null  object 
 3   LAT                 86584 non-null  float64
 4   LON                 86584 non-null  float64
 5   Precipitation       86584 non-null  float64
 6   LST                 46798 non-null  float64
 7   AAI                 73709 non-null  float64
 8   CloudFraction       73709 non-null  float64
 9   NO2_strat           73709 non-null  float64
 10  NO2_total           73709 non-null  float64
 11  NO2_trop            51111 non-null  float64
 12  TropopausePressure  73709 non-null  float64
 13  GT_NO2              82051 non-null  float64
dtypes: float64(11), object(3)
memory usage: 9.2+ MB


In [4]:
train = train[~train['GT_NO2'].isna()][:1000]# deleting some points because they don't have the target variable and we can't train the model without targets

In [5]:
train.info() # now we have only the data points with targets

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 0 to 1042
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID_Zindi            1000 non-null   object 
 1   Date                1000 non-null   object 
 2   ID                  1000 non-null   object 
 3   LAT                 1000 non-null   float64
 4   LON                 1000 non-null   float64
 5   Precipitation       1000 non-null   float64
 6   LST                 599 non-null    float64
 7   AAI                 994 non-null    float64
 8   CloudFraction       994 non-null    float64
 9   NO2_strat           994 non-null    float64
 10  NO2_total           994 non-null    float64
 11  NO2_trop            538 non-null    float64
 12  TropopausePressure  994 non-null    float64
 13  GT_NO2              1000 non-null   float64
dtypes: float64(11), object(3)
memory usage: 117.2+ KB


In [6]:
train.head()

,ID_Zindi,Date,ID,LAT,LON,Precipitation,LST,AAI,CloudFraction,NO2_strat,NO2_total,NO2_trop,TropopausePressure,GT_NO2
0,ID_ENTGC7,1/1/19,PD01,45.601585,11.903551,0.000000,NaN,0.230527,0.559117,0.000024,0.000117,NaN,14440.82126,31.0
1,ID_8JCCXC,1/1/19,PD04,45.371005,11.840830,3.047342,NaN,-0.074006,0.869309,0.000024,0.000127,NaN,14441.79815,42.0
2,ID_V3136Z,1/1/19,RO01,45.045825,12.060869,0.000000,NaN,0.024470,0.674160,0.000024,0.000086,NaN,14437.38294,31.0
3,ID_KRVZDJ,1/1/19,RO02,45.104075,11.553241,1.200467,NaN,-0.010442,0.920054,0.000024,0.000124,NaN,14440.83831,30.0
4,ID_PR351A,1/1/19,RO03,45.038758,11.790152,1.274564,NaN,-0.176178,0.747464,0.000024,0.000116,NaN,14438.79037,58.0


In [7]:
train.describe()

,LAT,LON,Precipitation,LST,AAI,CloudFraction,NO2_strat,NO2_total,NO2_trop,TropopausePressure,GT_NO2
count,1000.000000,1000.000000,1000.000000,599.000000,994.000000,994.000000,994.000000,994.000000,538.000000,994.000000,1000.000000
mean,45.418995,10.034498,0.984774,280.506177,-0.733477,0.181647,0.000026,0.000243,0.000159,17880.021875,46.832900
std,0.228689,1.087894,1.850932,2.363306,0.379558,0.224094,0.000004,0.000222,0.000097,3011.060557,19.771958
min,44.924694,8.736497,0.000000,268.700000,-2.519505,0.000000,0.000017,-0.000009,0.000006,14399.916930,0.600000
25%,45.249544,9.195325,0.000000,279.160000,-0.981363,0.021695,0.000023,0.000118,0.000079,16676.466528,33.093750
50%,45.463753,9.601223,0.000000,280.620000,-0.722201,0.077054,0.000025,0.000192,0.000151,16728.180745,44.000000
75%,45.601232,10.795564,1.410635,282.130000,-0.564894,0.239991,0.000028,0.000290,0.000219,19334.604875,58.025000
max,45.889734,12.590682,11.733904,286.780000,0.516055,0.955316,0.000035,0.001755,0.000568,24340.759500,137.400000


In [8]:
train['ID'].nunique() # ID's are not unique

77

In [ ]:
train.drop(columns=['Date','ID_Zindi','ID'],inplace=True)

/tmp/ipykernel_12775/1458128098.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  train['Date'] = pd.to_datetime(train['Date'])


## Scaling and Filling NaN values

In [13]:
scaler = MinMaxScaler((-1,1))


In [14]:
for feature in train.columns:
    train[feature] = train[feature].fillna(train[feature].mean())
    train[feature] = scaler.fit_transform(train[[feature]])

In [15]:
train

,LAT,LON,Precipitation,LST,AAI,CloudFraction,NO2_strat,NO2_total,NO2_trop,TropopausePressure,GT_NO2,day,dayofyear,month,year
0,0.402825,0.643436,-1.000000,0.305993,0.811878,0.170538,-0.227513,-0.857173,-0.453247,-0.991770,-0.555556,-1.0,-1.000000,-1.0,-1.0
1,-0.075042,0.610889,-0.480592,0.305993,0.611234,0.819939,-0.216931,-0.845836,-0.453247,-0.991574,-0.394737,-1.0,-1.000000,-1.0,-1.0
2,-0.748961,0.725071,-1.000000,0.305993,0.676116,0.411387,-0.206349,-0.892089,-0.453247,-0.992462,-0.555556,-1.0,-1.000000,-1.0,-1.0
3,-0.628242,0.461655,-0.795385,0.305993,0.653114,0.926176,-0.195767,-0.849237,-0.453247,-0.991767,-0.570175,-1.0,-1.000000,-1.0,-1.0
4,-0.763607,0.584592,-0.782755,0.305993,0.543917,0.564851,-0.195767,-0.858306,-0.453247,-0.992179,-0.160819,-1.0,-1.000000,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,0.634886,0.365352,-1.000000,0.871681,0.215538,-1.000000,0.068783,-0.921450,-0.900443,-0.034658,-0.584795,1.0,-0.922156,-1.0,-1.0
1039,0.315733,0.454085,-1.000000,0.440265,0.186860,-1.000000,0.111111,-0.812961,-0.561472,-0.014201,-0.175439,1.0,-0.922156,-1.0,-1.0
1040,-0.465447,0.335698,-1.000000,0.400442,0.030443,-0.977346,0.142857,-0.739274,-0.330274,-0.016237,-0.307018,1.0,-0.922156,-1.0,-1.0
1041,0.054281,0.190432,-1.000000,0.477876,0.127937,-1.000000,0.100529,-0.814094,-0.565029,-0.016769,-0.497076,1.0,-0.922156,-1.0,-1.0


## Modeling

In [16]:
rf = RandomForestRegressor(random_state=42)
linear_regression = LinearRegression()
svm_linear = SVR(kernel='linear')
svm_poly = SVR(kernel='poly')

In [17]:
models = {
        'Random Forest':rf,
        'Linear Regression':linear_regression,
        'SVM Linear Kernel':svm_linear,
        'SVM Poly':svm_poly
    }


In [18]:
comparisons = {

}

## Validation

In [19]:
def validate(model,metric):
    train_val,test_val = train_test_split(train,train_size=0.75,random_state=42)
    model.fit(train_val.drop(columns='GT_NO2'),train_val['GT_NO2'])
    pred = model.predict(test_val.drop(columns='GT_NO2'))
    score = metric(pred,test_val['GT_NO2'])
    return score

In [20]:
metrics = {
    'MAPE': mean_absolute_percentage_error,
    'MAE': mean_absolute_error,
    'RMSE': root_mean_squared_error
}

In [21]:
for i in models:
    comparisons[i]={

    }
    for metric in metrics:
        val = validate(models[i],metrics[metric])
        comparisons[i][metric] = val

## Results

In [22]:
for model in comparisons:
    print(model,':')
    for metric in comparisons[model]:
        print('  ',metric,'mean across Folds:')
        print(f'        {np.array(comparisons[model][metric]).mean():.3f}')
        print('-'*30)
    print('='*30)

Random Forest :
   MAPE mean across Folds:
        0.595
------------------------------
   MAE mean across Folds:
        0.109
------------------------------
   RMSE mean across Folds:
        0.144
------------------------------
Linear Regression :
   MAPE mean across Folds:
        0.787
------------------------------
   MAE mean across Folds:
        0.186
------------------------------
   RMSE mean across Folds:
        0.248
------------------------------
SVM Linear Kernel :
   MAPE mean across Folds:
        0.612
------------------------------
   MAE mean across Folds:
        0.185
------------------------------
   RMSE mean across Folds:
        0.251
------------------------------
SVM Poly :
   MAPE mean across Folds:
        1.485
------------------------------
   MAE mean across Folds:
        0.164
------------------------------
   RMSE mean across Folds:
        0.220
------------------------------
